# F1 Aerodynamics GEM Training — Kaggle

Trains `F1AeroNet` (Gauge-Equivariant Mesh CNN) to predict surface pressure (Cp),
wall shear stress (WSS), and drag coefficient (Cd) on F1 car geometries.

**Cells in order:**
1. GPU / environment check
2. Install dependencies
3. Clone code from GitHub
4. Copy dataset from Kaggle input
5. Write Kaggle training config
6. Pre-compute Cd normalisation stats
7. Run training
8. Save checkpoints to `/kaggle/working/output`

## 1 — GPU / Environment Check

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("No GPU detected — training will be slow on CPU.")

print(f"PyTorch: {torch.__version__}")

# Capture version strings for dependency install below
TORCH_VER = torch.__version__.split('+')[0]   # e.g. "2.3.0"
CUDA_TAG  = f"cu{torch.version.cuda.replace('.', '')}" if torch.cuda.is_available() else "cpu"
print(f"\nInstall tag: torch-{TORCH_VER}+{CUDA_TAG}")

## 2 — Install Dependencies

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

# torch-geometric (core, no C extensions needed for basic use)
pip('torch-geometric')

# Optional compiled extensions — provide significant speedup on CUDA
pyg_url = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'
try:
    pip('torch-scatter', 'torch-sparse', '-f', pyg_url)
    print('torch-scatter / torch-sparse installed.')
except Exception as e:
    print(f'[WARN] Could not install C extensions ({e}). Falling back to pure-Python ops.')

# CFD mesh I/O and scientific stack
pip('pyvista', 'vtk', 'scipy', 'pyyaml', 'tqdm')

print('\nAll dependencies installed.')

## 3 — Clone Code from GitHub

In [ ]:
import os, subprocess

REPO_URL  = 'https://github.com/aumrawal/GDL_Cars.git'
REPO_DIR  = '/kaggle/working/GDL_Cars'

if not os.path.exists(REPO_DIR):
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    print('Clone complete.')
else:
    print(f'{REPO_DIR} already exists — pulling latest changes.')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Add repo to Python path so imports work
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('\nRepo contents:')
for f in sorted(os.listdir(REPO_DIR)):
    print(f'  {f}')

## 4 — Copy Dataset

In [ ]:
import shutil, os

SRC_DATASET = '/kaggle/input/datasets/aumrawal29/drivaernet-200'
DST_DATASET = '/kaggle/working/drivaernet_real'

if not os.path.exists(DST_DATASET):
    print(f'Copying dataset...\n  {SRC_DATASET}\n  → {DST_DATASET}')
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print('Copy complete.')
else:
    print(f'{DST_DATASET} already exists, skipping copy.')

# Verify structure
print('\nDataset top-level:')
for entry in sorted(os.listdir(DST_DATASET)):
    full = os.path.join(DST_DATASET, entry)
    if os.path.isdir(full):
        count = len(os.listdir(full))
        print(f'  {entry}/  ({count} items)')
    else:
        print(f'  {entry}')

mesh_dir = os.path.join(DST_DATASET, 'meshes')
if os.path.isdir(mesh_dir):
    vtps = [f for f in os.listdir(mesh_dir) if f.endswith('.vtp')]
    print(f'\nFound {len(vtps)} VTP meshes in {mesh_dir}')
else:
    print(f'[WARNING] No meshes/ directory found in {DST_DATASET}.')
    print('Expected: drivaernet_real/meshes/<design_id>.vtp')

## 5 — Write Kaggle Training Config

Overrides the base config with Kaggle-specific paths and **the 3 Cd-collapse fixes**:
- `cd` loss weight → **1.0** (equal weight with Cp/WSS — correct after y_cd z-score normalisation brings all loss scales to the same order)
- `cd` loss type → **mse** (was l1 — MSE gradient grows with error magnitude, fighting mean collapse)
- Cd z-score normalisation applied in dataset via `cd_stats.json` (handled in code)

In [ ]:
import yaml, os

CHECKPOINT_DIR = '/kaggle/working/GDL_Cars/runs/'

kaggle_cfg = {
    'data': {
        'data_root':    '/kaggle/working/drivaernet_real',
        'max_vertices': 50000,
        'rho':          1.225,
        'U_inf':        83.33,
        'L_ref':        5.0,
    },
    'model': {
        'in_channels':           9,       # 6 coords+U_inf+x_frac+z_frac → +3 vertex normals
        'layer_types':           [[16, 2], [16, 2], [24, 2], [32, 1], [16, 1]],
        'nonlin_samples':        5,
        'head_dropout':          0.1,
        'break_symmetry_final':  True,
    },
    'training': {
        'epochs':         200,
        'batch_size':     1,
        'lr':             1e-3,
        'weight_decay':   0.0,
        'T_0':            50,     # CosineAnnealingWarmRestarts: first cycle length
        'T_mult':         2,      # each restart doubles the cycle length
        'min_lr':         1e-6,
        'grad_clip':      1.0,
        'checkpoint_dir': CHECKPOINT_DIR,
        'log_every':      10,
        'loss_weights': {
            'cp':  1.0,
            'wss': 1.0,
            'cd':  1.0,   # equal weight — y_cd is z-scored so loss scale matches Cp/WSS
            'cl':  0.0,
        },
        'loss_type': {
            'cp':  'huber',
            'wss': 'huber',
            'cd':  'mse',   # MSE gradient grows with error, fights mean collapse
            'cl':  'l1',
        },
    },
    'eval': {
        'checkpoint': os.path.join(CHECKPOINT_DIR, 'best.pt'),
        'output_dir': '/kaggle/working/output',
        'save_vtk':   True,
    },
}

cfg_path = os.path.join(REPO_DIR, 'configs', 'f1_kaggle.yaml')
with open(cfg_path, 'w') as f:
    yaml.dump(kaggle_cfg, f, default_flow_style=False, sort_keys=False)

print(f'Config written to {cfg_path}')
print(yaml.dump(kaggle_cfg, default_flow_style=False, sort_keys=False))

## 6 — Pre-compute Cd Normalisation Stats (Fix 2)

The cache (.pt files) stores raw Cd values. This cell computes mean/std from the
**training split only** and saves them to `cd_stats.json`. The trainer then applies
z-score normalisation at load time so the model sees Cd targets with std ≈ 1.

This step also triggers initial VTP → PyG cache conversion if not already done.

In [ ]:
import os, json, shutil, torch, sys

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

DATA_ROOT  = '/kaggle/working/drivaernet_real'
STATS_PATH = os.path.join(DATA_ROOT, 'cd_stats.json')

# ── Stale cache clearing ────────────────────────────────────────────────────
# Feature format changed (CACHE_VERSION 1→2): vertex normals added, per-component
# WSS normalisation. Delete old .pt files so they are regenerated from VTPs.
processed_dir = os.path.join(DATA_ROOT, 'processed')
if os.path.exists(processed_dir):
    stale_count = 0
    for split in os.listdir(processed_dir):
        split_dir = os.path.join(processed_dir, split)
        if os.path.isdir(split_dir):
            for fname in os.listdir(split_dir):
                if fname.endswith('.pt'):
                    pt_path = os.path.join(split_dir, fname)
                    try:
                        d = torch.load(pt_path, weights_only=False)
                        version = getattr(d, 'cache_version', torch.tensor([0])).item()
                        if version < 2:
                            os.remove(pt_path)
                            stale_count += 1
                    except Exception:
                        os.remove(pt_path)
                        stale_count += 1
    if stale_count:
        print(f'Cleared {stale_count} stale .pt cache files (CACHE_VERSION < 2).')
    else:
        print('No stale cache files found.')

# ── Cd normalisation stats ──────────────────────────────────────────────────
from data.drivaernet_dataset import DrivAerNetDataset

if os.path.exists(STATS_PATH):
    with open(STATS_PATH) as f:
        s = json.load(f)
    print(f'Cd stats already exist — skipping computation.')
    print(f"  mean = {s['cd_mean']:.6f}")
    print(f"  std  = {s['cd_std']:.6f}")
else:
    print('Building training cache and computing Cd stats...')
    print('(This converts VTP files to .pt — may take a few minutes on first run.)\n')

    train_ds = DrivAerNetDataset(
        data_root=DATA_ROOT, split='train', normalize_cd=False
    )

    cd_vals = []
    for i in range(len(train_ds)):
        data = train_ds[i]
        if hasattr(data, 'y_cd') and data.y_cd is not None:
            cd_vals.append(float(data.y_cd.reshape(-1)[0]))
        if (i + 1) % 20 == 0 or (i + 1) == len(train_ds):
            print(f'  processed {i+1}/{len(train_ds)}')

    if len(cd_vals) >= 2:
        t        = torch.tensor(cd_vals, dtype=torch.float32)
        cd_mean  = float(t.mean())
        cd_std   = float(t.std())
        stats    = {'cd_mean': cd_mean, 'cd_std': cd_std}
        with open(STATS_PATH, 'w') as f:
            json.dump(stats, f, indent=2)
        print(f'\nCd stats saved to {STATS_PATH}')
        print(f'  mean = {cd_mean:.6f}')
        print(f'  std  = {cd_std:.6f}')
        print(f'  n    = {len(cd_vals)}')
        print(f'  range = [{min(cd_vals):.4f}, {max(cd_vals):.4f}]')
    else:
        print(f'[ERROR] Only {len(cd_vals)} Cd values found. Check that y_cd is populated in VTP files.')

## 7 — Run Training

In [ ]:
import sys, yaml, os

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)

from train.trainer import train

cfg_path = os.path.join(REPO_DIR, 'configs', 'f1_kaggle.yaml')
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

print(f'Starting training: {cfg["training"]["epochs"]} epochs')
print(f'Checkpoint dir   : {cfg["training"]["checkpoint_dir"]}')
print(f'Cd loss weight   : {cfg["training"]["loss_weights"]["cd"]}  (was 0.1)')
print(f'Cd loss type     : {cfg["training"]["loss_type"]["cd"]}  (was l1)')
print()

train(cfg)

## 8 — Save Checkpoints to Output

In [ ]:
import shutil, os

CHECKPOINT_DIR = '/kaggle/working/GDL_Cars/runs/'
OUTPUT_DIR     = '/kaggle/working/output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

for fname in ['best.pt', 'last.pt', 'training_log.csv']:
    src = os.path.join(CHECKPOINT_DIR, fname)
    dst = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Saved: {dst}  ({size_mb:.1f} MB)')
    else:
        print(f'[NOT FOUND] {src}')

# Also copy Cd stats so they can be reused in evaluation
stats_src = '/kaggle/working/drivaernet_real/cd_stats.json'
if os.path.exists(stats_src):
    shutil.copy2(stats_src, os.path.join(OUTPUT_DIR, 'cd_stats.json'))
    print(f'Saved: {OUTPUT_DIR}/cd_stats.json')

print(f'\nOutput directory contents:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

## (Optional) Cd Scatter Plot — Sanity Check

Re-run after training to verify Cd predictions have spread away from the mean.

> **Note:** after Fix 2, `y_cd` and `pred['cd']` are both in **normalised** space
> (z-scored, mean≈0, std≈1). To convert back to physical Cd, load `cd_stats.json`
> and apply `cd_physical = cd_norm * cd_std + cd_mean`.

In [ ]:
import os, glob, json, torch
import matplotlib.pyplot as plt
import numpy as np

# Load Cd stats for denormalisation
with open('/kaggle/working/drivaernet_real/cd_stats.json') as f:
    cd_stats = json.load(f)
CD_MEAN = cd_stats['cd_mean']
CD_STD  = cd_stats['cd_std']

# Load best checkpoint
CKPT   = '/kaggle/working/output/best.pt'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from models.f1_net import F1AeroNet
from data.drivaernet_dataset import DrivAerNetDataset

ckpt      = torch.load(CKPT, map_location='cpu', weights_only=False)
saved_cfg = ckpt.get('cfg', {})
model     = F1AeroNet.from_config(saved_cfg.get('model', {}))
model.load_state_dict(ckpt['model'])
model     = model.to(device).eval()

# Collect predictions
val_ds = DrivAerNetDataset(
    data_root='/kaggle/working/drivaernet_real', split='val'
)
val_ds.set_cd_stats(CD_MEAN, CD_STD)

results = {'train': [], 'val': []}
for split in ['train', 'val']:
    ds = DrivAerNetDataset(
        data_root='/kaggle/working/drivaernet_real', split=split
    )
    ds.set_cd_stats(CD_MEAN, CD_STD)
    with torch.no_grad():
        for i in range(len(ds)):
            data = ds[i].to(device)
            pred = model(
                x=data.x, edge_index=data.edge_index,
                angles=data.edge_angles, transporters=data.edge_transporters,
                batch=torch.zeros(data.x.shape[0], dtype=torch.long, device=device),
            )
            # Denormalise to physical Cd
            cd_pred = pred['cd'].cpu().item() * CD_STD + CD_MEAN
            cd_true = data.y_cd.cpu().item() * CD_STD + CD_MEAN
            results[split].append((cd_true, cd_pred))

# Plot
fig, ax = plt.subplots(figsize=(6, 6))
colours = {'train': '#4C72B0', 'val': '#DD8452'}
for split, pts in results.items():
    trues  = [p[0] for p in pts]
    preds  = [p[1] for p in pts]
    mae    = np.mean(np.abs(np.array(trues) - np.array(preds)))
    corr   = float(np.corrcoef(trues, preds)[0, 1]) if len(trues) > 2 else float('nan')
    ax.scatter(trues, preds, s=20, alpha=0.7, color=colours[split],
               label=f'{split} (n={len(pts)}, MAE={mae:.4f}, r={corr:.3f})')

all_true = [p[0] for pts in results.values() for p in pts]
lo, hi   = min(all_true) * 0.98, max(all_true) * 1.02
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect')
ax.set_xlabel('True Cd (physical)')
ax.set_ylabel('Predicted Cd (physical)')
ax.set_title(f"Cd Prediction (epoch={ckpt.get('epoch', '?')})")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/output/cd_scatter.png', dpi=150)
plt.show()
print('Scatter plot saved.')